# 05 · Zero-shot transfer to human VL tissue following ACL injury

Each rat-trained ResNet-50 is applied to the human patches **without retraining**, with its
own weights and its own rat normalisation statistics; `cnn_morph` reads the human
morphometrics workbook, standardised with the rat training statistics.

| human comparison | rat model | class 0 / class 1 |
|---|---|---|
| `uninjured_1mo` | Control vs VML | Uninjured / 1-month post-ACLR |
| `uninjured_4mo` | Control vs VML | Uninjured / 4-month post-ACLR |
| `1mo_4mo` | Poor vs Very poor | 4-month / 1-month post-ACLR |

F1 is reported for the uninjured group (the 4-month group in 4-month vs 1-month).
`scripts/human_eval.py --selftest` first re-scores the rat test set through the same code
path and requires the predictions to match the sweep's own.

**`RUN`** decides what this notebook does.
* With the original data on disk, `False` (default) reads the results of the run behind
  the manuscript and checks them; `True` re-trains into `runs/rerun`.
* On the synthetic data, `RUN` is `True`: there are no saved results to read, so the models
  are trained (into `runs/synthetic/rerun`; finished runs are skipped).

The manuscript's results folders are never overwritten, and nothing is written into the
dataset.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
RES = P.roots(BASE)["cnn"]
if RUN:
    for extra in (["--selftest"], []):
        subprocess.run([PY, "-u", str(HERE / "scripts" / "human_eval.py"), "--data-root", str(P.DATA),
                        "--results-root", str(RES), "--models", "cnn,cnn_morph", *extra], check=True)
from manuscript_numbers import compute
N = compute(BASE, models=[], tasks=[])
H = pd.DataFrame(N["human"]).T
H[["patch_f1", "wsi_f1", "patch_macro", "wsi_macro", "patch_acc", "wsi_acc", "auc", "wsi_called0", "wsi_true0", "n_wsi"]].round(3)

In [ ]:
best = H.wsi_f1.astype(float).idxmax()
print(f"best WSI-level F1: {H.loc[best, 'wsi_f1']:.3f} ({best})"
      + ("   (synthetic data)" if SYNTHETIC else "   <- the abstract's 'as high as 0.849'"))

`wsi_called0` is how many WSIs the model called class 0 (`wsi_true0` are). With the
original data, in 4-month vs 1-month every WSI is called 4-month by both models, so its F1
reflects no discrimination.